[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session3/notebooks/01_finetune.ipynb)

# session3 · 01 · 전이학습(파인튜닝) — YOLO11n 을 우리 혈액 데이터로

- **이 노트북에서 배우는 것**: 2회차에서 혈액 도말에 **빗나갔던 COCO YOLO11n** 을 BCCD 로 **파인튜닝하면 맞히게 된다**.
- **입력**: 없음 — BCCD(공개, MIT)를 코드로 내려받아 YOLO 형식으로 변환합니다.
- **출력**: 학습 곡선 · **전/후 비교** · conf 스윕 4장 · 클래스별 성능 (`outputs/`) + 가중치(`weights/`)

> **수업 진행**: **셀 4(학습)** 를 누르고 이론 강의로 넘어갑니다(무료 T4 기준 10~15분). 강의 후 돌아와 **셀 5~9** 를 실행합니다.
> **런타임 > 런타임 유형 변경 > GPU(T4)** 로 먼저 설정하세요. 학습이 실패해도 **백업 가중치**로 셀 6~8 이 돌아갑니다.
> 그림 안 글자는 폰트 호환을 위해 영문입니다.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (session2 와 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session3"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session3/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 셀 2 · 의존성 설치 + 버전/GPU 확인
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

print("\n[설치된 버전 — requirements.txt 핀 확인용]")
for mod in ["torch", "torchvision", "ultralytics", "numpy", "matplotlib", "PIL"]:
    try:
        m = __import__(mod)
        print("  -", ("Pillow" if mod == "PIL" else mod), ":", getattr(m, "__version__", "?"))
    except Exception as e:
        print("  -", mod, ": import 실패 (", e, ")")

# 셀 4의 학습 시간이 여기서 갈린다 — 학생이 미리 알아야 한다.
import torch
print("\n[GPU]")
if torch.cuda.is_available():
    print("  사용 가능:", torch.cuda.get_device_name(0), " -> 셀 4는 50 epoch (약 10~15분)")
else:
    print("  GPU 없음 -> 셀 4는 5 epoch (CPU, 결과 약함)")
    print("  런타임 > 런타임 유형 변경 > GPU(T4) 로 바꾸시길 권합니다.")

In [ ]:
# 셀 3 · BCCD 전체 준비 + VOC -> YOLO 변환 + bccd.yaml
# 공식 분할(train 205 / val 87)을 그대로 사용한다. 직접 나누지 않는다.
import os, glob, shutil, subprocess, collections
import xml.etree.ElementTree as ET

SRC = os.path.join("data", "_bccd_src")
if not os.path.isdir(SRC):
    print("BCCD shallow clone (364장)...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Shenggan/BCCD_Dataset", SRC], check=False)
B = os.path.join(SRC, "BCCD")
if not os.path.isdir(B):
    raise RuntimeError("BCCD 확보 실패(네트워크). 셀 3을 다시 ▶ 실행하세요.")

CLASSES = ["RBC", "WBC", "Platelets"]      # 0=RBC, 1=WBC, 2=Platelets (yaml 과 순서 일치)
CID = {c: i for i, c in enumerate(CLASSES)}


def split_ids(s):
    with open(os.path.join(B, "ImageSets", "Main", s + ".txt")) as f:
        return [l.strip() for l in f if l.strip()]


root = os.path.join("data", "bccd")
for s in ("train", "val"):
    os.makedirs(os.path.join(root, "images", s), exist_ok=True)
    os.makedirs(os.path.join(root, "labels", s), exist_ok=True)

n_clip, n_degen = 0, 0
counts = collections.Counter()
n_img = {}
for s in ("train", "val"):
    ids = split_ids(s)
    for i in ids:
        jpg = os.path.join(B, "JPEGImages", i + ".jpg")
        xml = os.path.join(B, "Annotations", i + ".xml")
        if not (os.path.exists(jpg) and os.path.exists(xml)):
            continue
        shutil.copy(jpg, os.path.join(root, "images", s, i + ".jpg"))
        r = ET.parse(xml).getroot()
        sz = r.find("size")
        W, H = int(sz.find("width").text), int(sz.find("height").text)
        lines = []
        for o in r.findall("object"):
            name = o.find("name").text
            if name not in CID:
                continue
            b = o.find("bndbox")
            x1, y1, x2, y2 = [float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
            # 이미지 밖으로 나간 좌표는 0~W / 0~H 로 clip
            cx1, cy1 = max(0.0, x1), max(0.0, y1)
            cx2, cy2 = min(float(W), x2), min(float(H), y2)
            if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
                n_clip += 1
            w, h = cx2 - cx1, cy2 - cy1
            if w <= 0 or h <= 0:      # 면적 0 어노테이션(BCCD 에 2개 존재) -> 학습 라벨에서 제외
                n_degen += 1
                continue
            lines.append("%d %.6f %.6f %.6f %.6f" % (
                CID[name], (cx1 + cx2) / 2.0 / W, (cy1 + cy2) / 2.0 / H, w / W, h / H))
            counts[name] += 1
        with open(os.path.join(root, "labels", s, i + ".txt"), "w") as f:
            f.write("\n".join(lines) + ("\n" if lines else ""))
    n_img[s] = len(glob.glob(os.path.join(root, "images", s, "*.jpg")))

# bccd.yaml — path 는 절대경로여야 ultralytics 가 찾는다
yaml_path = os.path.join("data", "bccd.yaml")
abs_root = os.path.abspath(root).replace("\\", "/")
with open(yaml_path, "w") as f:
    f.write("path: %s\n" % abs_root)
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("names:\n")
    for i, c in enumerate(CLASSES):
        f.write("  %d: %s\n" % (i, c))

print("이미지  : train", n_img["train"], "장 / val", n_img["val"], "장  (공식 분할 그대로)")
print("라벨 txt: train", len(glob.glob(os.path.join(root, "labels", "train", "*.txt"))),
      "/ val", len(glob.glob(os.path.join(root, "labels", "val", "*.txt"))))
print("학습(train+val)에 쓰는 박스:", {c: counts[c] for c in CLASSES})
print("경계 clip 된 박스:", n_clip, "| 면적 0 이라 제외한 박스:", n_degen)
print("bccd.yaml 생성:", yaml_path, " (path =", abs_root, ")")
print()
print("RBC 4155 : WBC 372 : Platelets 361 — 11배 불균형. 이게 나중에 성능 표에서 그대로 드러납니다.")

---
# ⏱️ 셀 4 — 이 셀을 누르고 **이론 강의로 넘어갑니다. 끝날 때까지 두세요.**
**무료 코랩 T4 기준 10~15분** 걸립니다. 중간에 아무것도 하지 않아도 됩니다.
강의가 끝나고 돌아와서 셀 5부터 이어서 실행하세요.
---

In [ ]:
# 셀 4 · ★ 학습 (누르고 이론 강의로 넘어가는 셀)
import os, shutil, torch
from ultralytics import YOLO

HAS_GPU = torch.cuda.is_available()
EPOCHS = 50 if HAS_GPU else 5
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    print("EPOCHS =", EPOCHS, "-> 예상 소요 약 10~15분 (T4 기준). 이 셀을 누르고 강의로 넘어가세요.")
else:
    print("[경고] GPU 가 없습니다 -> EPOCHS =", EPOCHS, "(CPU). 몇 분 걸리고 결과가 약합니다.")
    print("       런타임 > 런타임 유형 변경 > GPU(T4) 로 바꾸면 제대로 학습됩니다.")

model = YOLO("yolo11n.pt")          # 2회차에서 빗나갔던 바로 그 COCO 사전학습 모델

# [중요] ultralytics 8.4 는 '상대경로' project 를 RUNS_DIR/<task>/ 아래로 끼워넣는다(8.3 과 동작이 다름).
# 절대경로로 주어 저장 위치를 <PROJECT_ROOT>/runs/bccd 로 고정한다 (셀 5~8 이 이 경로를 참조).
RUNS_PROJECT = os.path.abspath("runs")
model.train(
    data="data/bccd.yaml",
    epochs=EPOCHS, imgsz=640, batch=16,
    project=RUNS_PROJECT, name="bccd", exist_ok=True,
    seed=0, plots=True, verbose=True,
)

print("\n학습 결과 폴더:", getattr(model.trainer, "save_dir", "?"))
# 학습기가 알려주는 실제 경로를 우선 사용(경로 가정 제거), 없으면 관례 경로
best = str(getattr(model.trainer, "best", "") or "") or os.path.join("runs", "bccd", "weights", "best.pt")
os.makedirs("weights", exist_ok=True)
if os.path.exists(best):
    shutil.copy(best, os.path.join("weights", "bccd_yolo11n.pt"))
    print("학습 완료 -> weights/bccd_yolo11n.pt 저장  (원본:", best, ")")
else:
    print("[주의] best.pt 를 찾지 못했습니다:", best, "— 셀 6은 백업 가중치로 진행합니다.")

In [ ]:
# 셀 5 · 학습 곡선 (box loss / mAP50 두 개만 — 초보자가 길을 잃지 않게)
import os, shutil, csv
import matplotlib.pyplot as plt

os.makedirs("outputs", exist_ok=True)
dst = os.path.join("outputs", "03_curves.png")
src_png = os.path.join("runs", "bccd", "results.png")

if os.path.exists(src_png):
    shutil.copy(src_png, dst)
    print("복사:", src_png, "->", dst)
else:
    csv_path = os.path.join("runs", "bccd", "results.csv")
    if not os.path.exists(csv_path):
        raise RuntimeError("학습 결과가 없습니다(results.png / results.csv). 셀 4를 먼저 ▶ 실행하세요.")
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    keys = {k.strip(): k for k in rows[0]}

    def col(name):
        return keys.get(name)

    ep = [float(r[col("epoch")]) for r in rows] if col("epoch") else list(range(len(rows)))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    k_loss, k_map = col("train/box_loss"), col("metrics/mAP50(B)")
    if k_loss:
        ax[0].plot(ep, [float(r[k_loss]) for r in rows])
    ax[0].set_title("box loss (down = learning)"); ax[0].set_xlabel("epoch")
    if k_map:
        ax[1].plot(ep, [float(r[k_map]) for r in rows])
    ax[1].set_title("mAP50 (up = learning)"); ax[1].set_xlabel("epoch")
    plt.tight_layout()
    plt.savefig(dst, dpi=130)
    plt.show()
    print("저장:", dst)

print("loss 는 내려가고 mAP 는 올라간다 — 이게 '배우고 있다'는 뜻입니다.")

In [ ]:
# 셀 6 · ★ 전/후 비교 (이 노트북의 클라이맥스)
# 대상: BloodImage_00011 — test split (학습·검증 어디에도 안 쓰인 이미지)
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMG = os.path.join("data", "_bccd_src", "BCCD", "JPEGImages", "BloodImage_00011.jpg")
if not os.path.exists(IMG):
    raise RuntimeError("비교 이미지가 없습니다: %s — 셀 3을 먼저 ▶ 실행하세요." % IMG)


def resolve_ft_weights():
    """파인튜닝 가중치 결정 — 라이브 보험 순서(스펙 §4 셀6)."""
    cands = [os.path.join("weights", "bccd_yolo11n.pt"),          # 1) 셀 4가 만든 것
             os.path.join("runs", "bccd", "weights", "best.pt"),   # 2) 학습 산출물
             os.path.join("weights", "bccd_yolo11n_pretrained.pt")]  # 3) 레포 커밋 백업
    for i, c in enumerate(cands):
        if os.path.exists(c):
            if i == 2:
                print("[백업] 사전 학습된 가중치로 진행합니다.")
            return c
    raise RuntimeError("파인튜닝 가중치를 찾을 수 없습니다. 셀 4(학습)를 실행하거나 "
                       "weights/bccd_yolo11n_pretrained.pt 를 준비하세요.")


FT_WEIGHTS = resolve_ft_weights()
print("사용 가중치:", FT_WEIGHTS)

CONF = 0.25
before = YOLO("yolo11n.pt")(IMG, conf=CONF, verbose=False)[0]     # COCO (2회차와 같은 실패)
after = YOLO(FT_WEIGHTS)(IMG, conf=CONF, verbose=False)[0]        # 파인튜닝 (맞힘)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(before.plot()[:, :, ::-1])
ax[0].set_title("BEFORE: COCO yolo11n  (boxes: %d)" % len(before.boxes)); ax[0].axis("off")
ax[1].imshow(after.plot()[:, :, ::-1])
ax[1].set_title("AFTER: fine-tuned on BCCD  (boxes: %d)" % len(after.boxes)); ax[1].axis("off")
fig.suptitle("BloodImage_00011 (test split - never trained/validated on)   |   conf=%.2f" % CONF)
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "03_before_after.png"), dpi=130)
plt.show()
print("저장: outputs/03_before_after.png")
print("학습에도 검증에도 안 쓴 이미지에서 맞혀야 진짜 실력입니다.")

In [ ]:
# 셀 7 · confidence 임계값 스윕 (HTML 슬라이더용 4장)
# 같은 이미지·같은 모델, conf 만 변경. 네 장의 크기·여백·DPI 가 완전히 동일해야 슬라이더에서 안 튄다.
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

IMG = os.path.join("data", "_bccd_src", "BCCD", "JPEGImages", "BloodImage_00011.jpg")
if not os.path.exists(IMG):
    raise RuntimeError("이미지가 없습니다: %s — 셀 3을 먼저 ▶ 실행하세요." % IMG)


def resolve_ft_weights():
    """파인튜닝 가중치 결정 — 라이브 보험 순서(스펙 §4 셀6)."""
    cands = [os.path.join("weights", "bccd_yolo11n.pt"),          # 1) 셀 4가 만든 것
             os.path.join("runs", "bccd", "weights", "best.pt"),   # 2) 학습 산출물
             os.path.join("weights", "bccd_yolo11n_pretrained.pt")]  # 3) 레포 커밋 백업
    for i, c in enumerate(cands):
        if os.path.exists(c):
            if i == 2:
                print("[백업] 사전 학습된 가중치로 진행합니다.")
            return c
    raise RuntimeError("파인튜닝 가중치를 찾을 수 없습니다. 셀 4(학습)를 실행하거나 "
                       "weights/bccd_yolo11n_pretrained.pt 를 준비하세요.")


FT_WEIGHTS = resolve_ft_weights()
model = YOLO(FT_WEIGHTS)                 # 한 번만 로드
os.makedirs("outputs", exist_ok=True)

FIGSIZE = (7.0, 5.5)                     # 하드코딩 — 4장 동일
DPI = 130                                # 하드코딩 — 4장 동일
for conf, tag in [(0.10, "010"), (0.25, "025"), (0.50, "050"), (0.70, "070")]:
    res = model(IMG, conf=conf, verbose=False)[0]
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title("conf=%.2f | boxes=%d" % (conf, len(res.boxes)))
    ax.axis("off")
    out = os.path.join("outputs", "03_conf_%s.png" % tag)
    plt.savefig(out, dpi=DPI)            # bbox_inches 쓰지 않음(크기 동일 보장)
    plt.close(fig)
    print(out, "-> boxes =", len(res.boxes))
print("conf 를 올릴수록 확신 있는 박스만 남습니다 — 임계값은 '정확도 vs 놓침'의 트레이드오프입니다.")

In [ ]:
# 셀 8 · 클래스별 성능 (불균형 회수)
import os, glob, collections
import matplotlib.pyplot as plt
from ultralytics import YOLO

if not os.path.exists(os.path.join("data", "bccd.yaml")):
    raise RuntimeError("data/bccd.yaml 이 없습니다. 셀 3을 먼저 ▶ 실행하세요.")


def resolve_ft_weights():
    """파인튜닝 가중치 결정 — 라이브 보험 순서(스펙 §4 셀6)."""
    cands = [os.path.join("weights", "bccd_yolo11n.pt"),          # 1) 셀 4가 만든 것
             os.path.join("runs", "bccd", "weights", "best.pt"),   # 2) 학습 산출물
             os.path.join("weights", "bccd_yolo11n_pretrained.pt")]  # 3) 레포 커밋 백업
    for i, c in enumerate(cands):
        if os.path.exists(c):
            if i == 2:
                print("[백업] 사전 학습된 가중치로 진행합니다.")
            return c
    raise RuntimeError("파인튜닝 가중치를 찾을 수 없습니다. 셀 4(학습)를 실행하거나 "
                       "weights/bccd_yolo11n_pretrained.pt 를 준비하세요.")


FT_WEIGHTS = resolve_ft_weights()
model = YOLO(FT_WEIGHTS)
m = model.val(data="data/bccd.yaml", split="val", verbose=False)

# 클래스별 학습 박스 수 (셀 3이 만든 라벨에서 직접 집계)
train_counts = collections.Counter()
for t in glob.glob(os.path.join("data", "bccd", "labels", "train", "*.txt")):
    with open(t) as f:
        for line in f:
            if line.strip():
                train_counts[int(line.split()[0])] += 1

names = model.names
try:
    idx = [int(i) for i in m.ap_class_index]
except Exception:
    idx = sorted(names)
ap50 = list(getattr(m.box, "ap50", []))
prec = list(getattr(m.box, "p", []))
rec = list(getattr(m.box, "r", []))


def val_at(arr, i):
    return ("%.3f" % arr[i]) if i < len(arr) else "-"


rows = []
for i, ci in enumerate(idx):
    cname = names[int(ci)] if int(ci) in names else str(ci)
    rows.append([cname, str(train_counts.get(int(ci), 0)), val_at(ap50, i), val_at(prec, i), val_at(rec, i)])

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.axis("off")
tbl = ax.table(cellText=rows,
               colLabels=["class", "train boxes", "mAP50", "precision", "recall"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.7)
fig.suptitle("Per-class performance on val  (more training boxes -> better)")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "03_per_class.png"), dpi=130)
plt.show()
print("저장: outputs/03_per_class.png")
for r_ in rows:
    print("  ", r_)
print()
print("Platelets 가 약합니다. 모델이 나빠서가 아니라 라벨이 361개뿐이라서입니다. — 4회차 주제.")

In [ ]:
# 셀 9 · 검증 / 요약
import os

FILES = ["03_curves.png", "03_before_after.png",
         "03_conf_010.png", "03_conf_025.png", "03_conf_050.png", "03_conf_070.png",
         "03_per_class.png"]
print("=" * 58)
print(" session3 · 파인튜닝 검증")
print("=" * 58)
ok = 0
for f in FILES:
    p = os.path.join("outputs", f)
    exists = os.path.exists(p)
    ok += exists
    print("  %-22s : %s" % (f, "있음" if exists else "없음"))
print("  %-22s : %s" % ("weights/bccd_yolo11n.pt",
                        "있음" if os.path.exists(os.path.join("weights", "bccd_yolo11n.pt")) else "없음(백업으로 진행 가능)"))
print("-" * 58)
print("  outputs 그림: %d / %d" % (ok, len(FILES)))
if ok < len(FILES):
    print("  [주의] 빠진 그림이 있습니다 — 해당 셀(5~8)을 다시 ▶ 실행하세요.")
print("=" * 58)
print("배운 것:")
print(" 1) 사전학습 모델은 '남의 도메인'에서 배운 것 — 우리 데이터로 이어 학습(파인튜닝)하면 맞히게 된다.")
print(" 2) 학습·검증에 안 쓴 이미지(test)에서 맞혀야 진짜 실력이다(일반화).")
print(" 3) 라벨이 적은 클래스(Platelets 361개)는 성능이 낮다 — 모델 문제가 아니라 데이터 문제다.")